In [1]:

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM is not installed. XGBoost models will still run.")

TRAIN_PATH = "../data/train.csv"
TARGET = "Will_Buy_EV"
ID_COL = "id"

train = pd.read_csv(TRAIN_PATH)

y = (
    train[TARGET]
    .astype(str)
    .str.strip()
    .map({"No": 0, "Yes": 1})
    .astype(int)
)

X = train.drop(columns=[TARGET, ID_COL]).copy()

print("Train shape:", X.shape)
print("Positive rate:", y.mean())


Train shape: (668665, 13)
Positive rate: 0.17464500160768098


In [2]:

# ============================================================
# ORIGINAL FEATURE GROUPS
# ============================================================

numeric_cols = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level"
]

categorical_cols = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level"
]

# Fixed split for direct comparison with previous experiments.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)


Training: (534932, 13)
Validation: (133733, 13)


In [3]:

# ============================================================
# LEAKAGE-SAFE TARGET / FREQUENCY ENCODING
# ============================================================

def keyify(series):
    return series.astype("string").fillna("__MISSING__")

def fit_mapping(series, target, smoothing):
    key = keyify(series)

    tmp = pd.DataFrame({
        "key": key,
        "target": target.to_numpy()
    })

    stats = tmp.groupby("key")["target"].agg(["count", "mean"])

    global_mean = float(target.mean())

    encoded = (
        stats["count"] * stats["mean"]
        + smoothing * global_mean
    ) / (stats["count"] + smoothing)

    return encoded, global_mean

def apply_mapping(series, mapping, global_mean):
    return (
        keyify(series)
        .map(mapping)
        .fillna(global_mean)
        .astype(float)
    )

def fit_frequency(series):
    return keyify(series).value_counts(normalize=True)

def apply_frequency(series, mapping):
    return (
        keyify(series)
        .map(mapping)
        .fillna(0.0)
        .astype(float)
    )

def add_oof_encoding(
    X_tr,
    y_tr,
    X_va,
    columns,
    prefix,
    smoothing=20,
    n_splits=5
):
    X_tr = X_tr.copy()
    X_va = X_va.copy()

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for col in columns:

        oof_target = np.zeros(len(X_tr), dtype=np.float32)

        for fit_idx, fold_idx in skf.split(X_tr, y_tr):

            mapping, global_mean = fit_mapping(
                X_tr.iloc[fit_idx][col],
                y_tr.iloc[fit_idx],
                smoothing
            )

            oof_target[fold_idx] = apply_mapping(
                X_tr.iloc[fold_idx][col],
                mapping,
                global_mean
            ).to_numpy()

        X_tr[f"{prefix}_{col}_te"] = oof_target

        full_mapping, full_global = fit_mapping(
            X_tr[col],
            y_tr,
            smoothing
        )

        X_va[f"{prefix}_{col}_te"] = apply_mapping(
            X_va[col],
            full_mapping,
            full_global
        ).to_numpy()

        freq_mapping = fit_frequency(X_tr[col])

        X_tr[f"{prefix}_{col}_freq"] = apply_frequency(
            X_tr[col],
            freq_mapping
        )

        X_va[f"{prefix}_{col}_freq"] = apply_frequency(
            X_va[col],
            freq_mapping
        )

    return X_tr, X_va


In [4]:

# ============================================================
# DIGIT DECOMPOSITION
# ============================================================

digit_source_cols = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work"
]

def add_digit_features(df):
    out = df.copy()

    for col in digit_source_cols:

        values = pd.to_numeric(
            out[col],
            errors="coerce"
        )

        safe = (
            values
            .fillna(0)
            .abs()
            .astype(np.int64)
        )

        strings = safe.astype(str)

        out[f"{col}_last1"] = (
            safe % 10
        ).astype(np.int16)

        out[f"{col}_last2"] = (
            safe % 100
        ).astype(np.int32)

        out[f"{col}_last3"] = (
            safe % 1000
        ).astype(np.int32)

        out[f"{col}_first_digit"] = (
            strings.str[0]
            .astype(int)
        )

        out[f"{col}_digit_count"] = (
            strings.str.len()
            .astype(np.int16)
        )

        out[f"{col}_digit_sum"] = (
            strings.apply(
                lambda s: sum(int(c) for c in s)
            )
            .astype(np.int16)
        )

        out[f"{col}_first_last"] = (
            strings.str[0].astype(str)
            + "_"
            + (safe % 10).astype(str)
        ).astype("string")

        missing = values.isna()

        digit_key_cols = [
            f"{col}_last1",
            f"{col}_last2",
            f"{col}_last3",
            f"{col}_first_digit",
            f"{col}_digit_count",
            f"{col}_digit_sum",
            f"{col}_first_last"
        ]

        for key_col in digit_key_cols:
            if out[key_col].dtype == "object" or str(out[key_col].dtype) == "string":
                out.loc[missing, key_col] = "__MISSING__"

    return out

X_train = add_digit_features(X_train)
X_valid = add_digit_features(X_valid)

digit_cols = [
    c for c in X_train.columns
    if any(
        token in c
        for token in [
            "_last1",
            "_last2",
            "_last3",
            "_first_digit",
            "_digit_count",
            "_digit_sum",
            "_first_last"
        ]
    )
]

print("Digit-derived columns:", len(digit_cols))


Digit-derived columns: 35


In [5]:

# ============================================================
# GENERATOR-STYLE DISCRETE FEATURES
# ============================================================

def add_generator_features(df):
    out = df.copy()

    # Infrastructure totals
    out["Total_Charging_Stations"] = (
        out["Charging_Stations_Near_Home"]
        + out["Charging_Stations_Near_Work"]
    )

    out["Charging_Station_Gap"] = (
        out["Charging_Stations_Near_Home"]
        - out["Charging_Stations_Near_Work"]
    )

    # Ratios
    out["Income_Per_Commute"] = (
        out["Annual_Income_USD"]
        / (out["Daily_Commute_km"] + 1.0)
    )

    out["Income_Per_Age"] = (
        out["Annual_Income_USD"]
        / (out["Age"] + 1.0)
    )

    out["Commute_Per_Car"] = (
        out["Daily_Commute_km"]
        / (out["Number_of_Cars_Owned"] + 1.0)
    )

    out["Stations_Per_Car"] = (
        out["Total_Charging_Stations"]
        / (out["Number_of_Cars_Owned"] + 1.0)
    )

    # Strong behavioral interactions
    out["Commute_Charging_Interaction"] = (
        out["Daily_Commute_km"]
        * out["Total_Charging_Stations"]
    )

    out["Environmental_Commute"] = (
        out["Environmental_Concern_Level"]
        * out["Daily_Commute_km"]
    )

    out["Environmental_Charging"] = (
        out["Environmental_Concern_Level"]
        * out["Total_Charging_Stations"]
    )

    # Quantized numeric identities
    out["Income_10k"] = (
        out["Annual_Income_USD"] // 10000
    ).astype("Int64").astype("string")

    out["Income_5k"] = (
        out["Annual_Income_USD"] // 5000
    ).astype("Int64").astype("string")

    out["Commute_5km"] = (
        out["Daily_Commute_km"] // 5
    ).astype("Int64").astype("string")

    out["Commute_10km"] = (
        out["Daily_Commute_km"] // 10
    ).astype("Int64").astype("string")

    out["Age_5yr"] = (
        out["Age"] // 5
    ).astype("Int64").astype("string")

    out["Age_10yr"] = (
        out["Age"] // 10
    ).astype("Int64").astype("string")

    out["Home_Work_Stations_Pair"] = (
        keyify(out["Charging_Stations_Near_Home"])
        + "|"
        + keyify(out["Charging_Stations_Near_Work"])
    )

    out["Income_Commute_Pair"] = (
        keyify(out["Income_10k"])
        + "|"
        + keyify(out["Commute_5km"])
    )

    out["Age_Income_Pair"] = (
        keyify(out["Age_5yr"])
        + "|"
        + keyify(out["Income_10k"])
    )

    out["Age_Commute_Pair"] = (
        keyify(out["Age_5yr"])
        + "|"
        + keyify(out["Commute_5km"])
    )

    return out

X_train = add_generator_features(X_train)
X_valid = add_generator_features(X_valid)

generator_cat_cols = [
    "Income_10k",
    "Income_5k",
    "Commute_5km",
    "Commute_10km",
    "Age_5yr",
    "Age_10yr",
    "Home_Work_Stations_Pair",
    "Income_Commute_Pair",
    "Age_Income_Pair",
    "Age_Commute_Pair"
]

generator_num_cols = [
    "Total_Charging_Stations",
    "Charging_Station_Gap",
    "Income_Per_Commute",
    "Income_Per_Age",
    "Commute_Per_Car",
    "Stations_Per_Car",
    "Commute_Charging_Interaction",
    "Environmental_Commute",
    "Environmental_Charging"
]


In [6]:

# ============================================================
# PAIR + TRIPLE IDENTITIES
# ============================================================

def make_group(df, columns):
    result = keyify(df[columns[0]])

    for col in columns[1:]:
        result = result + "|" + keyify(df[col])

    return result

pair_definitions = [
    ["Age", "Daily_Commute_km"],
    ["Age", "Annual_Income_USD"],
    ["Annual_Income_USD", "Daily_Commute_km"],
    ["Charging_Stations_Near_Home", "Charging_Stations_Near_Work"],
    ["Daily_Commute_km", "Charging_Stations_Near_Home"],
    ["Daily_Commute_km", "Charging_Stations_Near_Work"],
    ["Number_of_Cars_Owned", "Daily_Commute_km"],
    ["Environmental_Concern_Level", "Daily_Commute_km"],
    ["Environmental_Concern_Level", "Annual_Income_USD"]
]

triple_definitions = [
    ["Age", "Daily_Commute_km", "Number_of_Cars_Owned"],
    ["Age", "Annual_Income_USD", "Daily_Commute_km"],
    ["Annual_Income_USD", "Daily_Commute_km", "Environmental_Concern_Level"],
    ["Daily_Commute_km", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"],
    ["Age", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"]
]

pair_cols = []
triple_cols = []

for i, cols in enumerate(pair_definitions):
    name = f"pair_{i}"
    X_train[name] = make_group(X_train, cols)
    X_valid[name] = make_group(X_valid, cols)
    pair_cols.append(name)

for i, cols in enumerate(triple_definitions):
    name = f"triple_{i}"
    X_train[name] = make_group(X_train, cols)
    X_valid[name] = make_group(X_valid, cols)
    triple_cols.append(name)

# Triple TE is specifically one of the public techniques
# reported to produce a material gain on this competition.
X_train, X_valid = add_oof_encoding(
    X_train,
    y_train,
    X_valid,
    pair_cols,
    prefix="pair",
    smoothing=40,
    n_splits=5
)

X_train, X_valid = add_oof_encoding(
    X_train,
    y_train,
    X_valid,
    triple_cols,
    prefix="triple",
    smoothing=50,
    n_splits=5
)

print("Pair groups:", len(pair_cols))
print("Triple groups:", len(triple_cols))


Pair groups: 9
Triple groups: 5


In [7]:

# ============================================================
# EXACT VALUE ENCODING
# ============================================================

# Exact-value encoding is the central signal identified in
# previous experiments and independently reported by public
# S6E9 analyses.

X_train, X_valid = add_oof_encoding(
    X_train,
    y_train,
    X_valid,
    numeric_cols,
    prefix="exact",
    smoothing=20,
    n_splits=5
)

# Digit-derived identities also get leakage-safe TE.
X_train, X_valid = add_oof_encoding(
    X_train,
    y_train,
    X_valid,
    digit_cols,
    prefix="digit",
    smoothing=25,
    n_splits=5
)

# Generator-style discrete identities.
X_train, X_valid = add_oof_encoding(
    X_train,
    y_train,
    X_valid,
    generator_cat_cols,
    prefix="generator",
    smoothing=30,
    n_splits=5
)

print("Feature matrix before preprocessing:", X_train.shape)


Feature matrix before preprocessing: (534932, 213)


In [8]:

# ============================================================
# NUMERIC / CATEGORICAL MODEL MATRIX
# ============================================================

encoded_numeric_cols = []

for col in numeric_cols:
    encoded_numeric_cols.extend([
        f"exact_{col}_te",
        f"exact_{col}_freq"
    ])

for col in digit_cols:
    encoded_numeric_cols.extend([
        f"digit_{col}_te",
        f"digit_{col}_freq"
    ])

for col in pair_cols:
    encoded_numeric_cols.extend([
        f"pair_{col}_te",
        f"pair_{col}_freq"
    ])

for col in triple_cols:
    encoded_numeric_cols.extend([
        f"triple_{col}_te",
        f"triple_{col}_freq"
    ])

for col in generator_cat_cols:
    encoded_numeric_cols.extend([
        f"generator_{col}_te",
        f"generator_{col}_freq"
    ])

model_numeric_cols = (
    numeric_cols
    + generator_num_cols
    + encoded_numeric_cols
)

all_categorical_cols = (
    categorical_cols
    + pair_cols
    + triple_cols
    + generator_cat_cols
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            model_numeric_cols
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            all_categorical_cols
        )
    ]
)

Xtr = preprocessor.fit_transform(X_train)
Xva = preprocessor.transform(X_valid)

print("Encoded train:", Xtr.shape)
print("Encoded valid:", Xva.shape)


Encoded train: (534932, 1734024)
Encoded valid: (133733, 1734024)


In [9]:

# ============================================================
# MODEL 1: LIGHTGBM
# ============================================================

predictions = {}
scores = []

if LIGHTGBM_AVAILABLE:

    lgb_configs = [
        {
            "name": "LGB_1",
            "n_estimators": 1400,
            "learning_rate": 0.025,
            "num_leaves": 31,
            "max_depth": -1,
            "min_child_samples": 80,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.05,
            "reg_lambda": 1.0
        },
        {
            "name": "LGB_2",
            "n_estimators": 1200,
            "learning_rate": 0.03,
            "num_leaves": 24,
            "max_depth": -1,
            "min_child_samples": 60,
            "subsample": 0.95,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0
        },
        {
            "name": "LGB_3",
            "n_estimators": 1600,
            "learning_rate": 0.02,
            "num_leaves": 40,
            "max_depth": -1,
            "min_child_samples": 100,
            "subsample": 0.90,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.05,
            "reg_lambda": 1.5
        }
    ]

    for cfg in lgb_configs:

        print("\nTraining", cfg["name"])

        model = LGBMClassifier(
            objective="binary",
            n_estimators=cfg["n_estimators"],
            learning_rate=cfg["learning_rate"],
            num_leaves=cfg["num_leaves"],
            max_depth=cfg["max_depth"],
            min_child_samples=cfg["min_child_samples"],
            subsample=cfg["subsample"],
            colsample_bytree=cfg["colsample_bytree"],
            reg_alpha=cfg["reg_alpha"],
            reg_lambda=cfg["reg_lambda"],
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        )

        model.fit(Xtr, y_train)

        pred = model.predict_proba(Xva)[:, 1]
        score = roc_auc_score(y_valid, pred)

        predictions[cfg["name"]] = pred
        scores.append((cfg["name"], score))

        print(f"{cfg['name']}: {score:.6f}")

else:
    print("Skipping LightGBM because it is unavailable.")



Training LGB_1
LGB_1: 0.944953

Training LGB_2
LGB_2: 0.944931

Training LGB_3
LGB_3: 0.944889


In [10]:

# ============================================================
# MODEL 2: XGBOOST COMPLEMENT
# ============================================================

xgb_configs = [
    {
        "name": "XGB_1",
        "n_estimators": 1200,
        "max_depth": 5,
        "learning_rate": 0.03,
        "min_child_weight": 2,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0
    },
    {
        "name": "XGB_2",
        "n_estimators": 1400,
        "max_depth": 4,
        "learning_rate": 0.025,
        "min_child_weight": 2,
        "subsample": 0.95,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0
    }
]

for cfg in xgb_configs:

    print("\nTraining", cfg["name"])

    model = XGBClassifier(
        n_estimators=cfg["n_estimators"],
        max_depth=cfg["max_depth"],
        learning_rate=cfg["learning_rate"],
        min_child_weight=cfg["min_child_weight"],
        subsample=cfg["subsample"],
        colsample_bytree=cfg["colsample_bytree"],
        reg_alpha=cfg["reg_alpha"],
        reg_lambda=cfg["reg_lambda"],
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(Xtr, y_train)

    pred = model.predict_proba(Xva)[:, 1]
    score = roc_auc_score(y_valid, pred)

    predictions[cfg["name"]] = pred
    scores.append((cfg["name"], score))

    print(f"{cfg['name']}: {score:.6f}")



Training XGB_1
XGB_1: 0.944977

Training XGB_2
XGB_2: 0.945044


In [11]:

# ============================================================
# RANK BLENDING
# ============================================================

def rank_average(pred_list):
    ranks = []

    for pred in pred_list:
        ranks.append(
            pd.Series(pred).rank(method="average").to_numpy()
            / len(pred)
        )

    return np.mean(ranks, axis=0)

score_df = (
    pd.DataFrame(scores, columns=["Model", "ROC_AUC"])
    .sort_values("ROC_AUC", ascending=False)
    .reset_index(drop=True)
)

print("\nINDIVIDUAL MODELS")
print(score_df.to_string(index=False))

rank_candidates = []

model_names = score_df["Model"].tolist()

if len(model_names) >= 2:
    for i in range(min(len(model_names), 4)):
        for j in range(i + 1, min(len(model_names), 4)):

            a = model_names[i]
            b = model_names[j]

            pred = rank_average([
                predictions[a],
                predictions[b]
            ])

            score = roc_auc_score(
                y_valid,
                pred
            )

            rank_candidates.append(
                (f"rank_{a}_{b}", score)
            )

if len(model_names) >= 3:

    top3 = model_names[:3]

    pred = rank_average([
        predictions[top3[0]],
        predictions[top3[1]],
        predictions[top3[2]]
    ])

    score = roc_auc_score(
        y_valid,
        pred
    )

    rank_candidates.append(
        ("rank_top3", score)
    )

rank_df = (
    pd.DataFrame(
        rank_candidates,
        columns=["Blend", "ROC_AUC"]
    )
    .sort_values("ROC_AUC", ascending=False)
    .reset_index(drop=True)
)

print("\nRANK BLENDS")

if len(rank_df):
    print(rank_df.to_string(index=False))
else:
    print("No rank blends available.")



INDIVIDUAL MODELS
Model  ROC_AUC
XGB_2 0.945044
XGB_1 0.944977
LGB_1 0.944953
LGB_2 0.944931
LGB_3 0.944889

RANK BLENDS
           Blend  ROC_AUC
rank_XGB_2_LGB_1 0.945135
       rank_top3 0.945120
rank_XGB_2_LGB_2 0.945100
rank_XGB_1_LGB_1 0.945084
rank_XGB_1_LGB_2 0.945053
rank_XGB_2_XGB_1 0.945044
rank_LGB_1_LGB_2 0.944987


In [12]:

# ============================================================
# FINAL EXPERIMENT 38 REPORT
# ============================================================

best_single = score_df.iloc[0]

best_score = float(best_single["ROC_AUC"])
best_name = best_single["Model"]

if len(rank_df):

    best_rank = rank_df.iloc[0]

    if float(best_rank["ROC_AUC"]) > best_score:
        best_score = float(best_rank["ROC_AUC"])
        best_name = best_rank["Blend"]

print("\n" + "=" * 75)
print("EXPERIMENT 38")
print("=" * 75)

print(f"Best configuration: {best_name}")
print(f"Best ROC-AUC:       {best_score:.6f}")

print("\nBenchmarks")
print(f"35A:                0.945084")
print(f"23B:                0.945243")
print(f"33B:                0.945331")
print(f"0.950 target:       0.950000")
print(f"0.960 target:       0.960000")

print("\nDeltas")
print(f"vs 35A:             {best_score - 0.945084:+.6f}")
print(f"vs 23B:             {best_score - 0.945243:+.6f}")
print(f"vs 0.950:           {best_score - 0.950000:+.6f}")
print(f"vs 0.960:           {best_score - 0.960000:+.6f}")

if best_score >= 0.960000:
    print("\n>>> 0.960+ TARGET HIT <<<")
elif best_score >= 0.950000:
    print("\n>>> 0.950+ TARGET HIT, 0.960 NOT YET REACHED <<<")
else:
    print("\n>>> 0.950 TARGET NOT REACHED <<<")



EXPERIMENT 38
Best configuration: rank_XGB_2_LGB_1
Best ROC-AUC:       0.945135

Benchmarks
35A:                0.945084
23B:                0.945243
33B:                0.945331
0.950 target:       0.950000
0.960 target:       0.960000

Deltas
vs 35A:             +0.000051
vs 23B:             -0.000108
vs 0.950:           -0.004865
vs 0.960:           -0.014865

>>> 0.950 TARGET NOT REACHED <<<
